# Előkészítés — futtasd le a gyakorlat ELŐTT

**MI alapú ember-gép interakció · Emberi emóciók felismerése**

Ez a notebook mindent előkészít, ami a 90 perces gyakorlathoz kell.
Nagyjából **10 perc**, és **egyszer kell lefuttatni**. Órán már csak
csatlakoztatjuk a Drive-ot, és mehetünk.

Mit csinál:

1. ellenőrzi, hogy GPU-s futtatókörnyezetben vagy
2. telepíti a szükséges csomagokat
3. letölti a segédmodult (`ser_utils.py`)
4. csatolja a Google Drive-ot, és oda tölti le a RAVDESS adatbázist (~208 MB)
5. letölti a WavLM modellt a Drive-on lévő gyorsítótárba (~380 MB)
6. lefuttat egy rövid öntesztet

> **Fontos:** a Colab menüjében állítsd át:
> **Futtatókörnyezet → Futtatókörnyezet típusának módosítása → T4 GPU**

## 1 · Futtatókörnyezet ellenőrzése

In [ ]:
import sys, os
print("Python:", sys.version.split()[0])
try:
    import torch
    print("torch:", torch.__version__, "| CUDA elérhető:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("\nNINCS GPU. Állítsd át: Futtatókörnyezet → Futtatókörnyezet "
              "típusának módosítása → T4 GPU, majd futtasd újra ezt a cellát.")
except ImportError:
    print("A torch még nincs telepítve — a következő cella megoldja.")

## 2 · Csomagok telepítése

Ez tart a legtovább, kb. 2-3 perc.

In [ ]:
%pip install -q opensmile transformers datasets soundfile librosa scikit-image
print("Kész.")

## 3 · A segédmodul letöltése

A `ser_utils.py` tartalmazza a letöltő, jellemzőkinyerő és kiértékelő függvényeket. Ezt nem az órán írjuk meg — az órai idő a lényegre megy el.

In [ ]:
# --- a segédmodul betöltése ---------------------------------------------
# Ha a letöltés nem megy, töltsd fel a ser_utils.py-t kézzel a bal oldali
# fájlpanelen (mappa ikon → feltöltés), és futtasd újra ezt a cellát.
import os, sys, subprocess, importlib

SER_UTILS_URL = "https://<a-tárgy-honlapja>/ser_utils.py"

def betolt_ser_utils(url=SER_UTILS_URL, fajl="ser_utils.py"):
    megvan = os.path.exists(fajl) and os.path.getsize(fajl) > 1000
    if not megvan:
        if "<" in url:
            raise RuntimeError(
                "A SER_UTILS_URL még helyőrző cím.\n"
                "Írd át a tényleges letöltési helyre, VAGY töltsd fel a "
                "ser_utils.py-t kézzel a bal oldali fájlpanelen, majd futtasd "
                "újra ezt a cellát.")
        ideiglenes = fajl + ".letoltes"
        kod = subprocess.run(["wget", "-q", "-O", ideiglenes, url]).returncode
        meret = os.path.getsize(ideiglenes) if os.path.exists(ideiglenes) else 0
        if kod != 0 or meret < 1000:
            if os.path.exists(ideiglenes):
                os.remove(ideiglenes)
            raise RuntimeError(
                f"A letöltés nem sikerult: {url} (wget kód: {kod}, {meret} bájt).\n"
                "Töltsd fel a ser_utils.py-t kézzel a bal oldali fájlpanelen, "
                "majd futtasd újra ezt a cellát.")
        os.replace(ideiglenes, fajl)

    szoveg = open(fajl, encoding="utf-8").read()
    if "def gpu_info" not in szoveg:
        raise RuntimeError(
            f"A {fajl} létezik ({len(szoveg)} bájt), de nem a segédmodul van "
            "benne. Töröld a fájlt, és töltsd fel újra.")

    sys.modules.pop("ser_utils", None)   # a korábbi (esetleg üres) modul eldobása
    return importlib.import_module("ser_utils")

su = betolt_ser_utils()
su.self_check()

## 4 · Drive csatolása és az adat letöltése

A RAVDESS a Drive-ra kerül, hogy órán ne kelljen újra letölteni. Az engedélykérő ablakban add meg a hozzáférést.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = "/content/drive/MyDrive/ser_gyakorlat"
os.makedirs(WORK, exist_ok=True)

# a HuggingFace gyorsítótár is a Drive-ra kerül, hogy a modell megmaradjon
os.environ["HF_HOME"] = os.path.join(WORK, "hf")

wav_root = su.download_ravdess(WORK)

## 5 · A modell előmelegítése

A WavLM súlyait most letöltjük a Drive-on lévő gyorsítótárba, hogy órán azonnal elérhető legyen.

In [ ]:
from transformers import AutoFeatureExtractor, AutoModel
MODEL = "microsoft/wavlm-base-plus"
_ = AutoFeatureExtractor.from_pretrained(MODEL)
_ = AutoModel.from_pretrained(MODEL)
print("A modell a gyorsítótárban van.")

## 6 · Önteszt

Ha ez a cella hibátlanul lefut, órán minden működni fog.

In [ ]:
import numpy as np

df = su.scan_ravdess(wav_root)
print("felvételek:", len(df), "| színészek:", df.actor.nunique(),
      "| érzelmek:", df.emotion.nunique())

# 8 felvétel végigfuttatása a teljes láncon
mini = df.sample(8, random_state=0)
waves = su.load_all(mini.path.tolist(), show_progress=False)
Xh, names = su.egemaps_features(waves, show_progress=False)
E = su.wavlm_embeddings(waves, layers=6, show_progress=False)

print(f"kézi jellemzők:  {Xh.shape}")
print(f"WavLM embedding: {E.shape}")
print("\nMINDEN RENDBEN — készen állsz a gyakorlatra.")

---

### Ha valami nem működött

| tünet | mit tegyél |
|---|---|
| „NINCS GPU” | Futtatókörnyezet → típus módosítása → **T4 GPU**, majd újra az 1. cellát |
| `AttributeError: module 'ser_utils' has no attribute ...` | a `ser_utils.py` üres vagy csonka. Töltsd fel kézzel a bal oldali fájlpanelen, majd futtasd újra a 3. cellát — az kidobja a hibás modult a cache-ből |
| „A SER_UTILS_URL még helyőrző cím” | a `ser_utils.py`-t kézzel kell feltölteni a bal oldali fájlpanelen (mappa ikon → feltöltés), aztán újra a 3. cella |
| a letöltés megszakad | futtasd újra a 4. cellát — a már letöltött rész megmarad |
| `ModuleNotFoundError` | futtasd újra a 2. cellát, majd **Futtatókörnyezet → Újraindítás** |
| a Drive nem csatolódik | Futtatókörnyezet → Újraindítás, majd újra a 4. cella |

Ha órakezdésig nem sikerül, ne aggódj: a gyakorlat közösen zajlik, és a kódot
a vetítőn is látni fogod.